In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
    
    def forward(self, predictions, targets):
        # Flatten
        predictions = predictions.view(-1)
        targets = targets.view(-1)
        
        # Calculate intersection and union
        intersection = (predictions * targets).sum()
        total = predictions.sum() + targets.sum()
        
        # Dice coefficient
        dice = (2.0 * intersection + self.smooth) / (total + self.smooth)
        
        # Dice loss
        loss = 1.0 - dice
        return loss

class CombinedLoss(nn.Module):
    def __init__(self, weight_ce=0.5, weight_dice=0.5):
        super(CombinedLoss, self).__init__()
        self.ce_loss = nn.BCEWithLogitsLoss()
        self.dice_loss = DiceLoss()
        self.weight_ce = weight_ce
        self.weight_dice = weight_dice
    
    def forward(self, predictions, targets):
        ce = self.ce_loss(predictions, targets.float())
        dice = self.dice_loss(torch.sigmoid(predictions), targets.float())
        combined = self.weight_ce * ce + self.weight_dice * dice
        return combined

# IoU metric for segmentation
def calculate_iou(predictions, targets, threshold=0.5):
    predictions = (predictions > threshold).float()
    intersection = (predictions * targets).sum()
    union = (predictions + targets).sum() - intersection
    iou = intersection / (union + 1e-8)
    return iou.item()

print("Loss functions and metrics defined")

## 2. Dice Loss for Segmentation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, init_features=32):
        super(UNet, self).__init__()
        features = init_features
        
        # Encoder
        self.encoder1 = self._block(in_channels, features, name="enc1")
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.encoder2 = self._block(features, features * 2, name="enc2")
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.encoder3 = self._block(features * 2, features * 4, name="enc3")
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Bottleneck
        self.bottleneck = self._block(features * 4, features * 8, name="bottleneck")
        
        # Decoder
        self.upconv3 = nn.ConvTranspose2d(features * 8, features * 4, kernel_size=2, stride=2)
        self.decoder3 = self._block(features * 8, features * 4, name="dec3")
        
        self.upconv2 = nn.ConvTranspose2d(features * 4, features * 2, kernel_size=2, stride=2)
        self.decoder2 = self._block(features * 4, features * 2, name="dec2")
        
        self.upconv1 = nn.ConvTranspose2d(features * 2, features, kernel_size=2, stride=2)
        self.decoder1 = self._block(features * 2, features, name="dec1")
        
        self.final_conv = nn.Conv2d(features, out_channels, kernel_size=1)
    
    def _block(self, in_channels, out_channels, name):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool1(enc1))
        enc3 = self.encoder3(self.pool2(enc2))
        
        # Bottleneck
        bottleneck = self.bottleneck(self.pool3(enc3))
        
        # Decoder
        dec3 = self.upconv3(bottleneck)
        dec3 = torch.cat([dec3, enc3], dim=1)
        dec3 = self.decoder3(dec3)
        
        dec2 = self.upconv2(dec3)
        dec2 = torch.cat([dec2, enc2], dim=1)
        dec2 = self.decoder2(dec2)
        
        dec1 = self.upconv1(dec2)
        dec1 = torch.cat([dec1, enc1], dim=1)
        dec1 = self.decoder1(dec1)
        
        output = self.final_conv(dec1)
        return output

model = UNet(in_channels=3, out_channels=1).to(device)
print("U-Net model created successfully")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 1. U-Net Architecture from Scratch

# Semantic Segmentation
## Pixel-level Classification for Image Segmentation

Topics covered:
- Encoder-Decoder architectures
- Dilated convolutions and receptive fields
- Skip connections and feature preservation
- U-Net architecture and variants
- Dice loss and Focal loss for segmentation
- Evaluation metrics (IoU, mIoU)
- Training on Cityscapes or custom datasets